# Write star-sited monomer SMILES

PolyzyMD builds dynamic polymer chains from **star-sited SMILES**. A monomer is
given as its repeat unit, as it is in the chain, with `*` on the two atoms that
bond to the neighbouring repeat units. A cap (end group) has one `*`.

This notebook helps you write them. Paste the SMILES of your monomer (for
example from PubChem), find the two linking atoms by their index, and copy the
star-sited SMILES into your config. Nothing here changes your config; check
each result by eye.

Run it in the PolyzyMD build environment:

```bash
pixi run -e build jupyter lab docs/source/tutorials/star_sited_monomers.ipynb
```

## 1. Draw your monomer

Paste the SMILES of the monomer. The default is methyl methacrylate (PubChem
CID 6658). Every atom is labelled with its index.

In [ ]:
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D


def show(mol, highlight=(), size=(520, 320)):
    """Draw a molecule with every atom labelled by its index."""
    drawer = rdMolDraw2D.MolDraw2DSVG(*size)
    options = drawer.drawOptions()
    options.addAtomIndices = True
    options.annotationFontScale = 0.8
    drawer.DrawMolecule(mol, highlightAtoms=list(highlight))
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()
    try:
        from IPython.display import SVG, display
    except ImportError:  # outside Jupyter: nothing to draw on
        return
    display(SVG(svg))


monomer_smiles = "C=C(C)C(=O)OC"  # methyl methacrylate; paste yours here

monomer = Chem.MolFromSmiles(monomer_smiles)
assert monomer is not None, f"RDKit cannot read {monomer_smiles!r}"
show(monomer)

## 2. Find the two linking atoms

In the chain, each repeat unit is bonded to the one before it and the one after
it. The two atoms that make those bonds carry the `*`.

- For a vinyl, acrylate or methacrylate monomer, they are the two carbons of
  the C=C double bond. Polymerization opens that bond: in the chain it is a
  single bond, and each of its carbons gains one bond to a neighbouring unit.
- The **first** `*` bonds toward the head of the chain, the second toward the
  tail. For methacrylates, PolyzyMD's examples put the first `*` on the CH2
  carbon, so the chain grows head to tail.

The cell below lists the C=C double bonds outside rings and carbonyls, the
usual polymerizable bond. It does not choose for you.

In [ ]:
candidates = []
for bond in monomer.GetBonds():
    a, b = bond.GetBeginAtom(), bond.GetEndAtom()
    if (
        bond.GetBondType() == Chem.BondType.DOUBLE
        and a.GetSymbol() == "C"
        and b.GetSymbol() == "C"
        and not bond.IsInRing()
    ):
        candidates.append((a.GetIdx(), b.GetIdx()))
        print(f"C=C between atoms {a.GetIdx()} and {b.GetIdx()}")
if not candidates:
    print("No C=C outside a ring: find the linking atoms from the chemistry of your polymer.")
show(monomer, highlight=[i for pair in candidates for i in pair])

## 3. Place the stars

Give the indices of the two linking atoms, head first. The helper turns a
double bond between them into a single bond, puts `*` on each, and redraws the
repeat unit with the two linking atoms highlighted. PolyzyMD then reads the
result the way a build will.

In [ ]:
head_atom, tail_atom = 0, 1  # the CH2 and the quaternary carbon of methyl methacrylate


def star_sited(mol, head, tail):
    """Return the repeat unit of ``mol`` with '*' on atoms ``head`` and ``tail``."""
    editable = Chem.RWMol(mol)
    bond = editable.GetBondBetweenAtoms(head, tail)
    if bond is not None and bond.GetBondType() == Chem.BondType.DOUBLE:
        bond.SetBondType(Chem.BondType.SINGLE)
    stars = []
    for atom in (head, tail):
        star = editable.AddAtom(Chem.Atom(0))
        editable.AddBond(atom, star, Chem.BondType.SINGLE)
        stars.append(star)
    unit = editable.GetMol()
    for atom in unit.GetAtoms():
        atom.SetNoImplicit(False)
        atom.SetNumExplicitHs(0)
    Chem.SanitizeMol(unit)
    # Start the SMILES at the head star, so the head '*' is written first.
    return Chem.MolToSmiles(unit, rootedAtAtom=stars[0])


repeat_unit = star_sited(monomer, head_atom, tail_atom)
print("Star-sited repeat unit:", repeat_unit)

from polyzymd.builders.polymer_generator import parse_star_smiles

checked = parse_star_smiles(repeat_unit, 2, "this monomer")
unit = Chem.MolFromSmiles(repeat_unit)
show(unit, highlight=[a.GetIdx() for a in unit.GetAtoms() if a.GetAtomicNum() == 0])

## 4. Choose the caps

Each chain end needs a cap: a SMILES with one `*` on the atom that bonds to the
chain. There is no default, because the caps are part of the molecule you
simulate.

- `[*][H]` caps the end with a hydrogen.
- `[*]C` caps it with a methyl group.
- Chains made by a controlled radical polymerization end in fragments of the
  initiator and the end group of that chemistry (for ATRP, an initiator
  fragment and a halide). Write those as SMILES with one `*` if you want them.

The cell prints a `type: polymer` entry to paste into `solutes:` of your
config. Fill in the probabilities, length and count of your study.

In [ ]:
head_cap = "[*][H]"
tail_cap = "[*][H]"
for name, cap in (("head", head_cap), ("tail", tail_cap)):
    parse_star_smiles(cap, 1, f"end_groups.{name}")

name = "MMA"  # a short monomer name
residue_name = "MMA"  # three characters, not an amino-acid, water or ion name

entry = f"""  - type: polymer
    generation_mode: "dynamic"
    type_prefix: "{name}"
    monomers:
      - label: "A"
        probability: 1.0
        name: "{name}"
        smiles: "{repeat_unit}"
        residue_name: "{residue_name}"
    end_groups:
      head: "{head_cap}"
      tail: "{tail_cap}"
    length: 5
    count: 3
    force_field: ["openff-2.3.0.offxml"]
    charges: {{method: nagl, model: "openff-gnn-am1bcc-1.0.0.pt"}}
"""
print(entry)

## 5. Build one short chain (optional)

This builds one chain of three repeat units with the PolyzyMD builder, charges
it with NAGL, relaxes it and checks it, as a build does. It takes a few
seconds. The files go into a temporary folder.

In [ ]:
import tempfile

from polyzymd.builders.polymer_generator import PolymerGenerator

with tempfile.TemporaryDirectory() as folder:
    generator = PolymerGenerator.from_smiles(
        [("A", name, residue_name, repeat_unit)],
        head=head_cap,
        tail=tail_cap,
        cache_directory=folder,
        force_field=["openff-2.3.0.offxml"],
        type_prefix=name,
    )
    chain = generator.generate_polymer("AAA")

total = sum(chain.partial_charges.m)
print(f"{chain.n_atoms} atoms, total charge {total:+.3f} e")
print("Residues:", sorted({(a.metadata['residue_number'], a.metadata['residue_name']) for a in chain.atoms}))
show(Chem.RemoveHs(chain.to_rdkit()), size=(700, 320))